# 02 — Sentinel-2 seasonal median composites and indices (M1.2)

Per-pixel median composites of the deduplicated, SCL-masked, offset-corrected Sentinel-2 L2A record over
`data/aoi/aoi.geojson`, one per Met Department season and season-year (FIM, SWM, SIM, NEM; 2016-2026 without the
partial SIM 2026 = 43 composites; ADR 0011). All logic is in `highland.stac`, `highland.masking`, `highland.indices`,
`highland.composites` and `highland.io`; this notebook orchestrates, tabulates and plots.

Decisions: SCL keep-set (class 2 kept, class 7 masked) ADR 0012; composite statistic, thin flag, index guard and layout
ADR 0013; terrain flag deferred to M1.3 ADR 0014. **The composites are not terrain-flagged yet.** Reflectance =
`(DN - 1000) / 10000` for processing baseline >= 04.00 (decided by baseline only), `DN / 10000` otherwise.
Indices are the median of per-scene indices; the per-band median reflectance is for display. Thin = `n_obs <= 2`.

In [ ]:
import importlib.metadata as md_
import logging
import os
import platform
import resource
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from highland import composites as C
from highland import stac
from highland.aoi import aoi_area_km2, load_aoi

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_rows", 100)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "outputs" / "M1.2"
OUT.mkdir(parents=True, exist_ok=True)
ITEMS_JSON = (
    ROOT / "outputs" / "M1.1" / "cache" / "items_s2_2015-06-01_2026-11-02.json"
)  # frozen M1.1 search (D7)
NUM_WORKERS = 4

## 1. AOI, frozen items, grid

In [ ]:
aoi = load_aoi()
raw_items = stac.load_items(ITEMS_JSON)
items, dropped = stac.dedupe_s2(raw_items)
geobox = stac.aoi_geobox(aoi, items, "s2", asset="B02", resolution=10.0)
ITEMS_SHA = C.items_hash(items)
print(
    f"AOI {aoi_area_km2(aoi):.1f} km2; {len(raw_items)} items -> {len(items)} after dedupe (dropped {len(dropped)})"
)
print(f"grid {geobox.shape} px, {geobox.crs}, 10 m; item-list sha256 {ITEMS_SHA[:16]}")

## 2. The 43 window-years: scenes, baselines, sun geometry

In [ ]:
rows = []
for season, sy in C.season_years():
    g = C.window_groups(items, season, sy)
    its = [i for v in g.values() for i in v]
    rows.append(
        {
            "season": season,
            "season_year": sy,
            "n_acquisitions": len(g),
            "baselines": ",".join(sorted({i.properties["s2:processing_baseline"] for i in its})),
            "sun_elev_min": round(min(90 - i.properties["s2:mean_solar_zenith"] for i in its), 1),
            "sun_elev_max": round(max(90 - i.properties["s2:mean_solar_zenith"] for i in its), 1),
        }
    )
windows = pd.DataFrame(rows)
print(len(windows), "window-years;", windows["n_acquisitions"].sum(), "acquisitions")
windows.pivot(index="season_year", columns="season", values="n_acquisitions")[list(C.SEASONS)]

## 3. Pilot: one thin and one full window-year (cold, scratch directory)
SIM 2023 (thin) and SWM 2023 (a full season). Wall time and the peak resident memory of this kernel so far.

In [ ]:
def peak_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


pilot = []
with tempfile.TemporaryDirectory(dir=OUT) as tmp:
    for season, sy in (("SIM", 2023), ("SWM", 2023)):
        t0 = time.perf_counter()
        C.composite_window(
            items, aoi, season, sy, Path(tmp), geobox=geobox, num_workers=NUM_WORKERS, force=True
        )
        pilot.append(
            {
                "window": f"{season} {sy}",
                "scenes": len(C.window_groups(items, season, sy)),
                "wall_s": round(time.perf_counter() - t0, 1),
                "peak_rss_mb": round(peak_mb()),
            }
        )
pd.DataFrame(pilot)

## 4. Full run (resumable: window-years whose item hash and parameters match are skipped)

In [ ]:
timings = []
t_all = time.perf_counter()
for season, sy in C.season_years():
    t0 = time.perf_counter()
    C.composite_window(items, aoi, season, sy, OUT, geobox=geobox, num_workers=NUM_WORKERS)
    timings.append(
        {"season": season, "season_year": sy, "wall_s": round(time.perf_counter() - t0, 1)}
    )
full_wall_s = time.perf_counter() - t_all
print(
    f"loop wall {full_wall_s:.0f} s (skipped window-years take ~0 s); peak RSS {peak_mb():.0f} MB"
)
size_gb = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file()) / 1e9
print(f"outputs/M1.2 size {size_gb:.2f} GB")

## 5. Summary per window-year; thin season-years against ADR 0011

In [ ]:
summary, scenes = C.collect_summaries(OUT)
assert len(summary) == 43, len(summary)
summary.to_csv(OUT / "composite_summary.csv", index=False)
scenes.to_csv(OUT / "scenes_used.csv", index=False)
cols = [
    "season",
    "season_year",
    "n_scenes",
    "n_scenes_with_clear",
    "baselines",
    "n_obs_median",
    "n_obs_p10",
    "share_zero_obs",
    "share_thin_1_2",
    "share_ge3_obs",
    "thin_season_year",
]
summary[cols].round(3)

In [ ]:
ADR0011_THIN = {
    ("FIM", 2016),
    ("FIM", 2017),
    ("NEM", 2016),
    ("SIM", 2019),
    ("SIM", 2023),
    ("SIM", 2020),
    ("FIM", 2022),
}
flag = set(
    map(tuple, summary.loc[summary["thin_season_year"], ["season", "season_year"]].to_numpy())
)
print("thin season-years (median n_obs <= 2):", sorted(flag, key=lambda t: (t[1], t[0])))
print("listed in ADR 0011 but not thin here:", sorted(ADR0011_THIN - flag))
print("thin here but not listed in ADR 0011:", sorted(flag - ADR0011_THIN))
print(
    "SIM thin in",
    int(summary.query("season == 'SIM'")["thin_season_year"].sum()),
    "of",
    int((summary["season"] == "SIM").sum()),
    "years",
)

In [ ]:
idx_cols = [
    "season",
    "season_year",
    "ndvi_p01",
    "ndvi_p50",
    "ndvi_p99",
    "ndvi_min",
    "ndvi_max",
    "ndvi_out_of_range_share",
    "evi_out_of_range_share",
    "ndmi_out_of_range_share",
    "nbr_out_of_range_share",
]
summary[idx_cols].round(4)

## 6. Maps and diagnostics (2D; no 3D drape here)

In [ ]:
def read(sy, win, kind, band):
    with rasterio.open(OUT / f"s2_{sy}_{win}_{kind}.tif") as src:
        names = list(src.descriptions)
        a = src.read(names.index(band) + 1).astype("float64")
        if src.nodata is not None and not np.isnan(src.nodata):
            a[a == src.nodata] = np.nan
        return a


show = [(2023, "SWM"), (2023, "SIM"), (2016, "FIM"), (2016, "NEM")]
fig, ax = plt.subplots(3, 4, figsize=(17, 12), constrained_layout=True)
for j, (sy, win) in enumerate(show):
    for i, (kind, band, cmap, lim) in enumerate(
        [
            ("indices", "ndvi", "viridis", (0, 1)),
            ("indices", "nbr", "viridis", (-0.2, 0.9)),
            ("obs", "n_obs", "magma", (0, 20)),
        ]
    ):
        im = ax[i, j].imshow(read(sy, win, kind, band), cmap=cmap, vmin=lim[0], vmax=lim[1])
        ax[i, j].set_title(f"{win} {sy} {band}")
        ax[i, j].set_axis_off()
        fig.colorbar(im, ax=ax[i, j], shrink=0.7)
fig.suptitle(
    "Median composites (not terrain-flagged; EPSG:32644, 10 m; pixels outside the AOI are blank)"
)
fig.savefig(OUT / "maps_ndvi_nbr_nobs.png", dpi=110)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)
x = pd.to_datetime(scenes["date_local"])
for k, (col, lab) in enumerate(
    [("scl_share_2", "SCL class 2 (cast shadow)"), ("scl_share_7", "SCL class 7 (unclassified)")]
):
    ax[k].scatter(
        x, 100 * scenes[col], s=6, c=scenes["baseline"].str[:2].astype(int), cmap="coolwarm"
    )
    ax[k].set_title(f"{lab}: share of AOI pixels per scene (%)")
    ax[k].set_yscale("symlog", linthresh=0.01)
ax[2].bar(
    range(len(summary)),
    100 * summary["share_le2_obs"],
    color=np.where(summary["thin_season_year"], "tab:red", "tab:blue"),
)
ax[2].set_title("share of pixels with n_obs <= 2 (%), red = thin season-year")
ax[2].set_xlabel("window-year index")
fig.savefig(OUT / "scl_class2_class7_and_thin.png", dpi=110)

In [ ]:
# How a composite's NDVI distribution compares with single scenes (SWM 2023, first six scenes with data).
g = C.window_groups(items, "SWM", 2023)
sel = {
    k: g[k]
    for k in scenes.query("season == 'SWM' and season_year == 2023 and clear_fraction > 0.3")[
        "acq_id"
    ].head(6)
}
st = C.scene_stack(sel, geobox).compute()
comp = read(2023, "SWM", "indices", "ndvi")
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(-0.2, 1.0, 100)
for t in range(st.sizes["time"]):
    v = st["ndvi"].isel(time=t).values.ravel()
    v = v[np.isfinite(v)]
    ax.hist(
        v, bins=bins, histtype="step", density=True, alpha=0.6, label=str(st["acq_id"].values[t])
    )
v = comp[np.isfinite(comp)]
ax.hist(v, bins=bins, histtype="step", density=True, color="k", lw=2, label="SWM 2023 composite")
ax.legend(fontsize=6)
ax.set_xlabel("NDVI")
ax.set_title("Composite vs single-scene NDVI (SWM 2023)")
fig.savefig(OUT / "ndvi_composite_vs_scenes.png", dpi=110)

## 7. README with the resolved environment

In [ ]:
env_txt = None
mm = shutil.which("micromamba") or os.environ.get("MAMBA_EXE")
if mm:
    r = subprocess.run(
        [mm, "env", "export", "--explicit", "-p", sys.prefix], capture_output=True, text=True
    )
    if r.returncode == 0 and r.stdout.strip():
        env_txt = r.stdout
if env_txt is None:
    raise RuntimeError(
        "micromamba not found: run 'micromamba env export --explicit' and save it to outputs/M1.2/environment.explicit.txt"
    )
(OUT / "environment.explicit.txt").write_text(env_txt)
vers = {
    p: md_.version(p)
    for p in (
        "rasterio",
        "odc-stac",
        "odc-geo",
        "numpy",
        "xarray",
        "dask",
        "pystac-client",
        "planetary-computer",
    )
}
vers["GDAL"], vers["PROJ"] = rasterio.__gdal_version__, rasterio.__proj_version__


def fmt(S_):
    return ", ".join(f"{s} {y}" for s, y in sorted(S_, key=lambda t: (t[1], t[0])))


thin_list = fmt(flag)
thin_extra = fmt(ADR0011_THIN - flag)
readme = f"""# outputs/M1.2 - Sentinel-2 seasonal median composites and indices

Produced by `notebooks/02_s2_composites.ipynb` (code: `highland.{{stac,masking,indices,composites,io}}`), task M1.2.
Everything here is git-ignored; rerun the notebook to regenerate (resumable). ADRs: 0012 (SCL keep-set), 0013 (composite
statistic, thin flag, index guard, layout), 0014 (terrain flag deferred to M1.3).

**These composites are NOT terrain-flagged.** Shaded slopes look less green; M1.3 adds the shadow share from the stored
per-scene sun geometry (`scenes_used.csv`) and clear masks (`clear_masks/`). Land cover is not masked either (M1.4/M1.6).
**Thin season-years must be read with the `thin` / `n_obs` band**: a median over 1-2 observations cannot reject undetected cloud.

## Inputs
- Frozen M1.1 item search `outputs/M1.1/cache/items_s2_2015-06-01_2026-11-02.json` ({len(raw_items)} items, {len(items)} after `dedupe_s2`),
  item-list sha256 `{ITEMS_SHA}`. Planetary Computer `sentinel-2-l2a`, tile 44NMN.
- AOI `data/aoi/aoi.geojson` ({aoi_area_km2(aoi):.1f} km2); grid {geobox.shape[0]} x {geobox.shape[1]} px, EPSG:32644, 10 m.
- Windows: FIM Mar-Apr, SWM May-Sep, SIM Oct-Nov, NEM Dec-Feb (December in the following season-year), season-years 2016-2026
  without the partial SIM 2026 = {len(summary)} composites.

## Method
- Reflectance (float32, 0-1): `(DN - 1000) / 10000` if `s2:processing_baseline` >= 04.00 else `DN / 10000`; DN 0 = no-data -> NaN;
  negatives kept. Decided by baseline only. odc-stac applies no scale/offset (verified against a direct rasterio read).
- 20 m bands (B8A, B11, B12) bilinear to 10 m; SCL nearest. SCL keep-set: classes 4, 5, 6 and 2 (cast shadow); 0, 1, 3, 7, 8-11 masked.
  A pixel is a valid observation if the SCL says clear and all seven bands are non-zero (source nodata 0 is forced for every asset, including the 20 m bands).
- NDVI/EVI use B08; NDMI/NBR use B8A (NDMI = (B8A-B11)/(B8A+B11), the moisture index). Indices are computed per scene,
  then the **median of the per-scene indices** is taken. NaN only where a denominator is exactly 0; nothing is clipped.
  Out-of-range shares are in `composite_summary.csv`. The reflectance bands are per-band medians for display only.

## Files (per window-year `s2_<season_year>_<WIN>_*`)
- `_indices.tif`: float32, bands ndvi, evi, ndmi, nbr; NaN nodata (also outside the AOI polygon).
- `_reflectance.tif`: float32, bands blue(B02), green(B03), red(B04), nir(B08), nir08(B8A), swir16(B11), swir22(B12); 0-1; NaN.
- `_obs.tif`: uint16 bands `n_obs`, `thin` (0 none, 1 = 1-2 obs, 2 = >= 3), `window_day_median` (median day of the window of the
  valid observations, 1 = first day of the window, 0 = none); nodata 65535 outside the AOI.
- `clear_masks/s2_<sy>_<WIN>.npz`: bit-packed per-scene 10 m valid masks (`unpack_clear_masks`), acq ids, sun azimuth/elevation.
- `scenes_used.csv`: per scene baseline, sun azimuth/elevation, window day, clear fraction, SCL class 0/2/7 shares (of AOI pixels).
  Source nodata is forced to 0 for every asset (some files declare none), so DN 0 / SCL 0 survive the warp.
- `composite_summary.csv`: per window-year n_obs median/p10, zero/thin shares, index value ranges and out-of-range shares.
- `meta/*.json` (resume keys and the rows behind the CSVs), `maps_ndvi_nbr_nobs.png`, `scl_class2_class7_and_thin.png`,
  `ndvi_composite_vs_scenes.png`, `environment.explicit.txt`.

## Thin season-years (median n_obs <= 2 over the AOI)
{thin_list}

Also flagged because ADR 0011 / the owner listed them (median n_obs is 3 but many pixels have <= 2 observations): {thin_extra}.
Read `share_le2_obs` per window in `composite_summary.csv`; no cut-off for dropping a window-year is applied.

**n_obs >= 3 does not guarantee a cloud-free median**: where most of a pixel's observations are contaminated by cloud SCL missed, the median is too. Diagnostic (ADR 0013, not a threshold): the share of AOI pixels with median blue > 0.15 is about 0.4 % in SWM composites but 3.5 % in SIM 2025, 3.7 % in FIM 2022, 2.1 % in SIM 2022 and 1.3 % in SIM 2024. Use `share_le2_obs` and cross-check against neighbouring years and other windows. EVI is not used for disturbance without a mask (ADR 0013).

## Run
Pilot (cold, scratch dir, from this notebook execution): {pilot}. Output size {size_gb:.2f} GB. The full loop in this execution resumed from existing files (0 s); cold full-run timings come from a scripted run and are appended below, labelled;
the peak RSS below is this notebook kernel's peak ({peak_mb():.0f} MB, pilot plus the resumed loop and the plots). Workers: {NUM_WORKERS} threads.
Versions: {vers}. Python {platform.python_version()}. Full lock: `environment.explicit.txt` (`micromamba env export --explicit`).

```
micromamba env create -n highland -f environment.yml      # or from environment.explicit.txt
pip install -e .
jupyter nbconvert --to notebook --execute notebooks/02_s2_composites.ipynb --output-dir <scratch> --ExecutePreprocessor.timeout=7200
```
"""
(OUT / "README.md").write_text(readme)
print(readme[:400])